# Trichinella ingestion and bronze loading

In [ ]:
import os
import json
import re
import shutil
import unicodedata
import pandas as pd

from pyspark.sql.functions import (
    col,
    lit,
    current_date,
    current_timestamp,
    regexp_extract,
    regexp_replace,
    to_date
)
from pyspark.sql.utils import AnalysisException



# PARAMETERS


# Estes valores são defaults.
# Quando corre via pipeline, devem ser substituídos pelos parâmetros do pipeline.
load_mode = "incremental"          # full ou incremental
run_id = "manual"



# CHECK DOS PARÂMETROS RECEBIDOS

print("Parâmetros recebidos pelo notebook:")
print("load_mode:", load_mode)
print("run_id:", run_id)

if load_mode not in ["full", "incremental"]:
    raise ValueError(f"load_mode inválido: {load_mode}")



# FUNÇÕES AUXILIARES

def clean_column_name(col_name):
    c = str(col_name).strip().lower()
    c = unicodedata.normalize("NFD", c).encode("ascii", "ignore").decode("utf-8")
    c = re.sub(r"[\s\-]+", "_", c)
    c = re.sub(r"[^a-z0-9_]", "", c)
    c = re.sub(r"_+", "_", c).strip("_")
    return c



# 1. DEFINIR PATHS

folder_name = "Trichinella"

local_input_path = f"/lakehouse/default/Files/{folder_name}"

spark_staging_path = f"Files/{folder_name}/CSV_Staging_Latest"
local_staging_path = f"/lakehouse/default/{spark_staging_path}"
log_file_path = f"{local_staging_path}/_processed_files_log.json"

staging_input_path = "Files/Trichinella/CSV_Staging_Latest/*/*/*.csv"

target_lakehouse = "lh_dsp10"
schema_name = "brz"
table_name = f"{target_lakehouse}.{schema_name}.trichinella"



# 2. PREPARAR STAGING E LOG

if load_mode == "full":
    print("Modo FULL: limpar staging antigo e ignorar log.")

    if os.path.exists(local_staging_path):
        shutil.rmtree(local_staging_path)

    os.makedirs(local_staging_path, exist_ok=True)

    processed_log = []

elif load_mode == "incremental":
    print("Modo INCREMENTAL: manter staging e ler log.")

    os.makedirs(local_staging_path, exist_ok=True)

    processed_log = []
    if os.path.exists(log_file_path):
        try:
            with open(log_file_path, "r") as f:
                processed_log = json.load(f)
        except Exception as e:
            print(f"Aviso: não foi possível ler o log. Erro: {e}")
            processed_log = []


new_processed_log = list(processed_log)
found_new_data = False


# 3. ENCONTRAR FICHEIROS EXCEL

excel_files = [
    f for f in os.listdir(local_input_path)
    if f.lower().endswith(".xlsx")
    and os.path.isfile(os.path.join(local_input_path, f))
]

print(f"Ficheiros Excel encontrados: {len(excel_files)}")



# 4. PROCESSAR FICHEIROS EXCEL PARA STAGING CSV

for file_name in excel_files:

    if file_name in processed_log:
        print(f"Ficheiro já processado, ignorado: {file_name}")
        continue

    file_path = os.path.join(local_input_path, file_name)
    xl = pd.ExcelFile(file_path)

    target_tabs = [
        sheet for sheet in xl.sheet_names
        if re.match(r"^20\d{2}$", str(sheet).strip())
    ]

    for sheet in target_tabs:
        print(f"Reading snapshot: {file_name} -> Tab {sheet}")

        pdf = pd.read_excel(file_path, sheet_name=sheet)

        if pdf.empty:
            continue

        pdf.columns = [str(c).strip() for c in pdf.columns]
        spark_df = spark.createDataFrame(pdf.astype(str))

        first_col = spark_df.columns[0]
        spark_df = spark_df.filter(
            (col(first_col).isNotNull()) &
            (col(first_col) != "nan") &
            (col(first_col) != "")
        )

        spark_df = (
            spark_df
            .withColumn("meta_source_file", lit(file_name))
            .withColumn("meta_source_sheet", lit(sheet))
            .withColumn("meta_ingestion_date", current_date().cast("string"))
            .withColumn("audit_run_id", lit(run_id))
            .withColumn("audit_load_mode", lit(load_mode))
        )

        output_dir = f"{spark_staging_path}/{file_name.replace('.xlsx', '')}/{sheet}"

        spark_df.coalesce(1).write \
            .mode("overwrite") \
            .option("header", "true") \
            .csv(output_dir)

    new_processed_log.append(file_name)
    found_new_data = True


# 5. ATUALIZAR LOG

if found_new_data:
    with open(log_file_path, "w") as f:
        json.dump(new_processed_log, f)

    print("New Trichinella snapshots processed to staging.")
else:
    print("All Trichinella files are up to date.")



# 6. LER STAGING E PREPARAR BRONZE

print("Step 1: Reading data from staging...")

try:
    df_staging = spark.read.option("header", "true").csv(staging_input_path)

except AnalysisException:
    raise ValueError("Não existem ficheiros CSV no staging para processar.")


print("Step 2: Cleaning headers...")

for old_col_name in df_staging.columns:
    new_col_name = clean_column_name(old_col_name)
    df_staging = df_staging.withColumnRenamed(old_col_name, new_col_name)


print("Step 3: Extracting source date from filename...")

df_bronze = (
    df_staging
    .withColumn(
        "meta_source_file_date",
        to_date(
            regexp_replace(
                regexp_extract(col("meta_source_file"), r"(\d{4}_\d{2}_\d{2})", 1),
                "_",
                "-"
            )
        )
    )
    .withColumn("audit_brz_load_timestamp", current_timestamp())
    .withColumn("audit_run_id", lit(run_id))
    .withColumn("audit_load_mode", lit(load_mode))
)

df_bronze = df_bronze.filter(col("meta_source_file_date").isNotNull())



# 7. DEFINIR DADOS A ESCREVER


if load_mode == "full":
    print("Modo FULL: todos os dados do staging serão escritos com overwrite.")
    df_bronze_new = df_bronze

elif load_mode == "incremental":
    print("Modo INCREMENTAL: aplicar left_anti contra Bronze existente.")

    try:
        df_existing = spark.table(table_name).select(
            "meta_source_file",
            "meta_source_sheet"
        ).distinct()

        df_bronze_new = df_bronze.join(
            df_existing,
            on=["meta_source_file", "meta_source_sheet"],
            how="left_anti"
        )

    except AnalysisException:
        print("Tabela Bronze ainda não existe. Todos os dados serão inseridos.")
        df_bronze_new = df_bronze



# 8. ESCREVER BRONZE

n_rows = df_bronze_new.count()

if n_rows > 0:

    if load_mode == "full":
        df_bronze_new.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(table_name)

        print(f"FULL load concluído. Tabela {table_name} reconstruída com {n_rows} linhas.")

    elif load_mode == "incremental":
        df_bronze_new.write \
            .format("delta") \
            .mode("append") \
            .option("mergeSchema", "true") \
            .saveAsTable(table_name)

        print(f"INCREMENTAL load concluído. Appended {n_rows} new rows.")

else:
    print("No new data to write to Bronze.")

# Validação

Foi feito um teste que adicionou coluna de validacao na tabela delta trichinella na camada bronze do lakehouse lh10dsp e o codigo de baixo apenas remove essa coluna

In [14]:
# # Tabela no lakehouse 12
# source_table = "lh_dsp12.brz.trichinella"

# # Coluna a remover
# coluna_remover = "teste_validacao_lakehouse"

# # 1. Ler tabela Delta do lakehouse 12
# df = spark.table(source_table)

# # 2. Remover coluna
# df_sem_coluna = df.drop(coluna_remover)

# # 3. Reescrever a mesma tabela no lakehouse 12
# df_sem_coluna.write \
#     .format("delta") \
#     .mode("overwrite") \
#     .option("overwriteSchema", "true") \
#     .saveAsTable(source_table)

# print(f"Coluna '{coluna_remover}' removida da tabela {source_table} com sucesso.")

StatementMeta(, ab4bb2cd-62b6-460d-8a41-677d9c61fef7, 16, Finished, Available, Finished, False)

Coluna 'teste_validacao_lakehouse' removida da tabela lh_dsp12.brz.trichinella com sucesso.


In [20]:
# from pyspark.sql import functions as F

# df = spark.read.table("lh_dsp10.brz.trichinella")
# print("Total rows in Trichinella Bronze:", df.count())
# print("Snapshots Summary:")
# df.select("meta_source_file_date", "meta_source_sheet", "meta_source_file") \
#     .distinct() \
#     .orderBy(F.desc("meta_source_file_date"), "meta_source_sheet") \
#     .show(truncate=False)

# # Preview
# display(df.limit(5))

StatementMeta(, a139b1a3-2a47-4308-b692-c48885ad9871, 22, Finished, Available, Finished, False)

Total rows in Trichinella Bronze: 610735
Snapshots Summary:
+---------------------+-----------------+------------------------------------------------------------+
|meta_source_file_date|meta_source_sheet|meta_source_file                                            |
+---------------------+-----------------+------------------------------------------------------------+
|2024-12-11           |2011             |Registo de pesquisa de Trichinella 2011-2015_2024_12_11.xlsx|
|2024-12-11           |2012             |Registo de pesquisa de Trichinella 2011-2015_2024_12_11.xlsx|
|2024-12-11           |2013             |Registo de pesquisa de Trichinella 2011-2015_2024_12_11.xlsx|
|2024-12-11           |2014             |Registo de pesquisa de Trichinella 2011-2015_2024_12_11.xlsx|
|2024-12-11           |2015             |Registo de pesquisa de Trichinella 2011-2015_2024_12_11.xlsx|
|2024-12-11           |2016             |Registo de pesquisa de Trichinella 2016-2020_2024_12_11.xlsx|
|2024-12-11  

SynapseWidget(Synapse.DataFrame, 855b2780-6b40-45b4-b457-c9608db3b4f0)

In [19]:
# from pyspark.sql import functions as F

# df = spark.read.table("lh_dsp10.brz.trichinella")
# df.groupBy("meta_source_sheet", "meta_source_file").count().orderBy("meta_source_sheet").show(truncate=False)
# display(df.limit(5))

StatementMeta(, a139b1a3-2a47-4308-b692-c48885ad9871, 21, Finished, Available, Finished, False)

+-----------------+------------------------------------------------------------+-----+
|meta_source_sheet|meta_source_file                                            |count|
+-----------------+------------------------------------------------------------+-----+
|2011             |Registo de pesquisa de Trichinella 2011-2015_2024_12_11.xlsx|50823|
|2012             |Registo de pesquisa de Trichinella 2011-2015_2024_12_11.xlsx|49019|
|2013             |Registo de pesquisa de Trichinella 2011-2015_2024_12_11.xlsx|45777|
|2014             |Registo de pesquisa de Trichinella 2011-2015_2024_12_11.xlsx|48024|
|2015             |Registo de pesquisa de Trichinella 2011-2015_2024_12_11.xlsx|48783|
|2016             |Registo de pesquisa de Trichinella 2016-2020_2024_12_11.xlsx|47014|
|2017             |Registo de pesquisa de Trichinella 2016-2020_2024_12_11.xlsx|45079|
|2018             |Registo de pesquisa de Trichinella 2016-2020_2024_12_11.xlsx|42984|
|2019             |Registo de pesquisa de T

SynapseWidget(Synapse.DataFrame, f2b43261-0244-476e-aa70-e57ff76b6159)